# Tier-1 Sparse Autoencoder Agents (Multi-Modality)
This notebook implements the Unsupervised Sparse LSTM Autoencoder strategy (with L1 penalty).

**Core Architecture Details:**
- **4 Independent Sparse Autoencoders** (Logon, Email, File, HTTP).
- **Deterministic Train/Val/Test Split:** 
  - Train: First 800 users (Normal behavior only)
  - Validation: Next 100 users (Used for Hyperparameter & Threshold Tuning)
  - Test: Final 100 users (Used ONLY for final evaluation)
- **MLflow Tracking:** Logs hyperparameters, metrics, and artifacts (ROC, Confusion Matrix).
- **Ablation Study:** Grid search over `hidden_dim` (16, 32, 64), `sequence_length` (7, 14), and `sparsity_weight` (0, 1e-2, 1e-3, 1e-4, 1e-5).
- **Regularization:** L1 Sparsity, Dropout, Weight Decay, and Early Stopping.


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, ConfusionMatrixDisplay
import mlflow
import itertools
import gc
from pathlib import Path
import os
import matplotlib.pyplot as plt

# Dynamic Path Resolution

# === Determinism: seed all RNGs for reproducible results (as in week3/tier1_score_generation.ipynb) ===
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Use a seeded generator for DataLoader shuffle
g = torch.Generator()
g.manual_seed(SEED)

print(f"All RNGs seeded with SEED={SEED}")

ROOT_DIR = Path.cwd()
while not (ROOT_DIR / "pyproject.toml").exists() and ROOT_DIR.parent != ROOT_DIR:
    ROOT_DIR = ROOT_DIR.parent

DATA_FILE = ROOT_DIR / "data" / "dayr4.2.csv"
MLFLOW_DB = f"sqlite:///{ROOT_DIR / 'mlflow.db'}"

mlflow.set_tracking_uri(MLFLOW_DB)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 1. Global Deterministic Split

In [2]:
print("Establishing Global User Split...")
df_users = pd.read_csv(DATA_FILE, usecols=['user'])
all_unique_users = sorted(df_users['user'].unique())

# Train: 800, Val: 100, Test: 100
TRAIN_USERS = set(all_unique_users[:800])
VAL_USERS = set(all_unique_users[800:900])
TEST_USERS = set(all_unique_users[900:1000])

print(f"Total Users: {len(all_unique_users)}")
print(f"Train Users: {len(TRAIN_USERS)}")
print(f"Val Users: {len(VAL_USERS)}")
print(f"Test Users: {len(TEST_USERS)}")
del df_users
gc.collect()

Establishing Global User Split...
Total Users: 1000
Train Users: 800
Val Users: 100
Test Users: 100


1754

## 2. Dataset & Model

In [3]:
class SequenceDataset(Dataset):
    def __init__(self, df, feature_cols, scaler, seq_len=14):
        self.features = scaler.transform(df[feature_cols].fillna(0).astype(np.float32))
        self.labels = (df['insider'].values > 0).astype(int)
        
        # Save raw data for exporting later
        self.users = df['user'].values
        self.weeks = df['week'].values
        self.days = df['day'].values
        
        self.seq_len = seq_len
        self.indices = []
        
        user_counts = df['user'].value_counts(sort=False)
        current_idx = 0
        for count in user_counts:
            if count >= seq_len:
                for i in range(count - seq_len + 1):
                    self.indices.append(current_idx + i)
            current_idx += count

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start_idx = self.indices[idx]
        seq = self.features[start_idx : start_idx + self.seq_len]
        label = self.labels[start_idx : start_idx + self.seq_len].max()
        
        # We need to return the user, week, day of the LAST day in the sequence for exporting
        end_idx = start_idx + self.seq_len - 1
        user_id = self.users[end_idx]
        week = self.weeks[end_idx]
        day = self.days[end_idx]
        
        return torch.tensor(seq), torch.tensor(seq), torch.tensor(label, dtype=torch.float32), user_id, week, day

class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, dropout=0.2):
        super(LSTMAutoencoder, self).__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.decoder = nn.LSTM(hidden_dim, input_dim, batch_first=True)
        
    def forward(self, x):
        _, (h, _) = self.encoder(x)
        h = self.dropout(h)
        latent = h[-1]
        seq_len = x.shape[1]
        h_repeated = latent.unsqueeze(1).repeat(1, seq_len, 1)
        out, _ = self.decoder(h_repeated)
        return out, latent


## 3. Hyperparameter Tuning & Export Loop

In [ ]:
modalities = ['logon', 'email', 'file_device', 'http']

param_grid = {
    'hidden_dim': [16, 32, 64],
    'sequence_length': [7, 14],
    'sparsity_weight': [0, 1e-2, 1e-3, 1e-4, 1e-5]
}

fixed_params = {
    'learning_rate': 0.001,
    'dropout': 0.2,
    'weight_decay': 1e-5,
    'epochs': 20, 
    'optimizer': 'Adam'
}

all_test_predictions = []

def get_optimal_threshold(mses, labels):
    best_f1 = 0
    best_thresh = 0
    for p in np.linspace(90, 99.9, 100):
        thresh = np.percentile(mses, p)
        preds = (mses > thresh).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh
    
    if best_thresh == 0:
        best_thresh = np.percentile(mses, 95)
        
    return best_thresh

for modality in modalities:
    print(f"\n{'='*50}")
    print(f"STARTING ABLATION STUDY FOR {modality.upper()}")
    print(f"{'='*50}")
    
    mlflow.set_experiment(f"SparseAutoencoder_{modality.capitalize()}")
    
    with open(DATA_FILE, 'r') as f:
        all_cols = f.readline().strip().split(',')
        
    feature_cols = [c for c in all_cols if modality in c or c in ['O', 'C', 'E', 'A', 'N', 'role']]
    usecols = ['user', 'day', 'week', 'insider'] + feature_cols
    
    print(f"Loading {len(feature_cols)} features for {modality}...")
    df = pd.read_csv(DATA_FILE, usecols=usecols)
    df = df.sort_values(by=['user', 'day'])
    
    train_df = df[df['user'].isin(TRAIN_USERS)]
    val_df = df[df['user'].isin(VAL_USERS)]
    test_df = df[df['user'].isin(TEST_USERS)]
    
    train_normal = train_df[train_df['insider'] == 0]
    
    scaler = StandardScaler()
    scaler.fit(train_normal[feature_cols].fillna(0).astype(np.float32))
    
    best_val_f1 = -1
    best_val_auc = -1
    best_model_state = None
    best_params = None
    best_threshold = 0
    best_seq_len = 14
    
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    for run_idx, params in enumerate(combinations):
        seq_len = params['sequence_length']
        hidden_dim = params['hidden_dim']
        sparsity_weight = params['sparsity_weight']
        
        print(f"\n--- Run {run_idx+1}/{len(combinations)} | Seq: {seq_len}, Hidden: {hidden_dim}, Sparsity: {sparsity_weight} ---")
        
        with mlflow.start_run():
            mlflow.log_params(params)
            mlflow.log_params(fixed_params)
            
            train_dataset = SequenceDataset(train_normal, feature_cols, scaler, seq_len=seq_len)
            train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, generator=g)
            
            val_dataset = SequenceDataset(val_df, feature_cols, scaler, seq_len=seq_len)
            val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)
            
            model = LSTMAutoencoder(input_dim=len(feature_cols), hidden_dim=hidden_dim, dropout=fixed_params['dropout']).to(device)
            optimizer = optim.Adam(model.parameters(), lr=fixed_params['learning_rate'], weight_decay=fixed_params['weight_decay'])
            criterion = nn.MSELoss()
            
            best_train_loss = float('inf')
            best_train_recon_loss = float('inf')
            best_train_l1_penalty = float('inf')
            patience = 3
            patience_counter = 0
            
            for epoch in range(fixed_params['epochs']):
                model.train()
                total_loss = 0
                total_recon_loss = 0
                total_l1_penalty = 0
                
                for batch_x, batch_y, _, _, _, _ in train_loader:
                    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                    optimizer.zero_grad()
                    out, latent = model(batch_x)
                    
                    recon_loss = criterion(out, batch_y)
                    l1_penalty = torch.mean(torch.abs(latent))
                    
                    loss = recon_loss + (sparsity_weight * l1_penalty)
                    loss.backward()
                    optimizer.step()
                    
                    total_loss += loss.item()
                    total_recon_loss += recon_loss.item()
                    total_l1_penalty += l1_penalty.item()
                
                avg_loss = total_loss / len(train_loader)
                avg_recon_loss = total_recon_loss / len(train_loader)
                avg_l1_penalty = total_l1_penalty / len(train_loader)
                
                if avg_loss < best_train_loss:
                    best_train_loss = avg_loss
                    best_train_recon_loss = avg_recon_loss
                    best_train_l1_penalty = avg_l1_penalty
                    patience_counter = 0
                else:
                    patience_counter += 1
                    
                if patience_counter >= patience:
                    print(f"Early stopping triggered at epoch {epoch+1}")
                    break
            
            mlflow.log_metric("train_total_loss", best_train_loss)
            mlflow.log_metric("train_recon_loss", best_train_recon_loss)
            mlflow.log_metric("train_l1_penalty", best_train_l1_penalty)
            
            model.eval()
            val_mses, val_labels = [], []
            with torch.no_grad():
                for batch_x, _, labels, _, _, _ in val_loader:
                    batch_x = batch_x.to(device)
                    out, _ = model(batch_x)
                    mse = torch.mean((out - batch_x)**2, dim=[1, 2]).cpu().numpy()
                    val_mses.extend(mse)
                    val_labels.extend(labels.numpy())
            
            threshold = get_optimal_threshold(val_mses, val_labels)
            val_preds = (val_mses > threshold).astype(int)
            
            val_auc = roc_auc_score(val_labels, val_mses)
            val_f1 = f1_score(val_labels, val_preds, zero_division=0)
            
            mlflow.log_metric("val_auc", val_auc)
            mlflow.log_metric("val_f1", val_f1)
            mlflow.log_metric("threshold", threshold)
            
            print(f"Val AUC: {val_auc:.4f} | Val F1: {val_f1:.4f} | Thresh: {threshold:.4f}")
            
            if val_f1 > best_val_f1 or (val_f1 == best_val_f1 and val_auc > best_val_auc):
                best_val_f1 = val_f1
                best_val_auc = val_auc
                best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
                best_params = params
                best_threshold = threshold
                best_seq_len = seq_len
                
    print(f"\n>> Best Params for {modality.upper()}: {best_params} | Val F1: {best_val_f1:.4f}")
    
    best_model = LSTMAutoencoder(input_dim=len(feature_cols), hidden_dim=best_params['hidden_dim'], dropout=fixed_params['dropout']).to(device)
    best_model.load_state_dict(best_model_state)
    best_model.eval()
    
    test_dataset = SequenceDataset(test_df, feature_cols, scaler, seq_len=best_seq_len)
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)
    
    test_mses, test_labels = [], []
    test_users, test_weeks, test_days = [], [], []
    
    with torch.no_grad():
        for batch_x, _, labels, u, w, d in test_loader:
            batch_x = batch_x.to(device)
            out, _ = best_model(batch_x)
            mse = torch.mean((out - batch_x)**2, dim=[1, 2]).cpu().numpy()
            test_mses.extend(mse)
            test_labels.extend(labels.numpy())
            test_users.extend(u)
            test_weeks.extend(w.numpy())
            test_days.extend(d.numpy())
            
    test_preds = (np.array(test_mses) > best_threshold).astype(int)
    test_auc = roc_auc_score(test_labels, test_mses)
    test_precision = precision_score(test_labels, test_preds, zero_division=0)
    test_recall = recall_score(test_labels, test_preds, zero_division=0)
    test_f1 = f1_score(test_labels, test_preds, zero_division=0)
    
    print(f"TEST METRICS -> AUC: {test_auc:.4f} | F1: {test_f1:.4f} | Precision: {test_precision:.4f} | Recall: {test_recall:.4f}")
    
    cm = confusion_matrix(test_labels, test_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    cm_path = ROOT_DIR / f"{modality}_sparse_confusion_matrix.png"
    plt.savefig(cm_path)
    plt.close()
    
    fpr, tpr, _ = roc_curve(test_labels, test_mses)
    plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {test_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Receiver Operating Characteristic - Sparse {modality.upper()}')
    plt.legend(loc="lower right")
    roc_path = ROOT_DIR / f"{modality}_sparse_roc_curve.png"
    plt.savefig(roc_path)
    plt.close()
    
    with mlflow.start_run(run_name=f"{modality}_best_test"):
        mlflow.log_params(best_params)
        mlflow.log_params(fixed_params)
        mlflow.log_metric("test_auc", test_auc)
        mlflow.log_metric("test_f1", test_f1)
        mlflow.log_metric("test_precision", test_precision)
        mlflow.log_metric("test_recall", test_recall)
        mlflow.log_artifact(cm_path)
        mlflow.log_artifact(roc_path)
        mlflow.set_tag("is_best_model", "true")
        
    os.remove(cm_path)
    os.remove(roc_path)
    
    def normalize_to_prob(mse_scores, threshold, steepness=10):
        return 1 / (1 + np.exp(-(mse_scores - threshold) * steepness))
        
    test_probs = normalize_to_prob(np.array(test_mses), best_threshold)
    
    df_preds = pd.DataFrame({
        'user': test_users,
        'week': test_weeks,
        'day': test_days,
        'label': test_labels,
        f'{modality}_score': test_probs
    })
    
    df_preds = df_preds.groupby(['user', 'week', 'day', 'label'], as_index=False).max()
    all_test_predictions.append(df_preds)
    
    del df, train_df, val_df, test_df, train_normal
    gc.collect()

print("\nMerging final scores...")
dfs_indexed = [df.set_index(['user', 'week', 'day', 'label']) for df in all_test_predictions]
tier1_scores = pd.concat(dfs_indexed, axis=1).reset_index()
tier1_scores.fillna(0, inplace=True)
out_path = ROOT_DIR / "week3" / "tier1_sparse_scores.csv"
tier1_scores.to_csv(out_path, index=False)
print(f"Saved tier1_sparse_scores.csv to {out_path}")



STARTING ABLATION STUDY FOR LOGON


2026/07/29 11:29:25 INFO mlflow.tracking.fluent: Experiment with name 'SparseAutoencoder_Logon' does not exist. Creating a new experiment.


Loading 21 features for logon...

--- Run 1/30 | Seq: 7, Hidden: 16, Sparsity: 0 ---


2026/07/29 11:29:39 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/07/29 11:29:39 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably

Val AUC: 0.4984 | Val F1: 0.0422 | Thresh: 6.7416

--- Run 2/30 | Seq: 7, Hidden: 16, Sparsity: 0.01 ---
Val AUC: 0.4954 | Val F1: 0.0422 | Thresh: 6.7159

--- Run 3/30 | Seq: 7, Hidden: 16, Sparsity: 0.001 ---
Val AUC: 0.4959 | Val F1: 0.0422 | Thresh: 6.6964

--- Run 4/30 | Seq: 7, Hidden: 16, Sparsity: 0.0001 ---
Val AUC: 0.4961 | Val F1: 0.0422 | Thresh: 6.6653

--- Run 5/30 | Seq: 7, Hidden: 16, Sparsity: 1e-05 ---
Val AUC: 0.4949 | Val F1: 0.0422 | Thresh: 6.7204

--- Run 6/30 | Seq: 14, Hidden: 16, Sparsity: 0 ---
Val AUC: 0.4955 | Val F1: 0.0411 | Thresh: 29.7892

--- Run 7/30 | Seq: 14, Hidden: 16, Sparsity: 0.01 ---
Val AUC: 0.4877 | Val F1: 0.0411 | Thresh: 29.7890

--- Run 8/30 | Seq: 14, Hidden: 16, Sparsity: 0.001 ---
Val AUC: 0.4855 | Val F1: 0.0411 | Thresh: 29.7650

--- Run 9/30 | Seq: 14, Hidden: 16, Sparsity: 0.0001 ---
Val AUC: 0.4856 | Val F1: 0.0411 | Thresh: 29.8487

--- Run 10/30 | Seq: 14, Hidden: 16, Sparsity: 1e-05 ---
Val AUC: 0.4873 | Val F1: 0.0411 | Thres